# Blackbox-explanation pipeline on POLAR simulated DTR data

Reproduces the SAE-based explanation pipeline from Explain_BB.pdf, adapted to the POLAR simulated 3-stage Dynamic Treatment Regime data.

**Setup**
- POLAR trajectory: `(s_1, a_1, s_2, a_2, s_3, a_3, s_4)`. `s_k \in [0,1]^2` are patient indicators, `a_k \in {0,1}` are treatment decisions, reward is at stage 3.
- **Blackbox M**: small MLP that maps the trajectory **through stage 3** (`s_1..s_3`, `a_1..a_3`) to a binary `outcome > median` label. Crucially, `s_4` is excluded from the input to keep the prediction non-trivial.
- **SAE**: top-k Sparse Autoencoder trained on M's second hidden layer.
- **LLM1**: proposes concept labels for each SAE feature using contrastive evidence sets.
- **Verifier**: short MATCH / NO_MATCH calls on held-out evidence.
- **LLM2**: structured explanation citing only active concepts.
- **Judge**: predicts the outcome from the explanation alone.
- **K-candidate selection**: K LLM2 outputs scored by `KL(P_M || P_Judge) + λ_spar |S| + λ_len Len(E)`.

**LLM backend**: `StubClient` by default (deterministic, no GPU). Switch to `QwenLocalClient` (cell 8) when you're ready to spend GPU time on real LLM calls.

## 0. Environment setup (auto-detect local vs Colab)

The next cell auto-detects whether you're on Google Colab. On Colab it
installs dependencies and clones the repo so the rest of the notebook runs
unchanged. Locally it's a no-op.

In [ ]:
import os, subprocess

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = 'https://github.com/syLe7elsup/RL_LLM_Pipeline.git'
COLAB_REPO_DIR = '/content/bb_pipeline'

if IN_COLAB:
    print('Detected Google Colab — bootstrapping environment...')
    # 1. Install Python deps
    subprocess.check_call(['pip', '-q', 'install',
                           'transformers', 'accelerate', 'sentence-transformers'])
    # 2. Clone (or pull) the repo
    if not os.path.isdir(COLAB_REPO_DIR):
        subprocess.check_call(['git', 'clone', REPO_URL, COLAB_REPO_DIR])
    else:
        subprocess.check_call(['git', '-C', COLAB_REPO_DIR, 'pull', '--ff-only'])
    # 3. Run from inside the repo so relative paths and `from bb_pipeline.X` work
    os.chdir(COLAB_REPO_DIR)
    print(f'cwd = {os.getcwd()}')
else:
    print('Local run — Colab bootstrap skipped.')

## 1. Imports + config

In [ ]:
import os, sys, json, warnings, time
import numpy as np
warnings.filterwarnings('ignore')

# Add the parent directory to sys.path so we can import bb_pipeline.*
sys.path.insert(0, os.path.abspath('..'))

from bb_pipeline.polar_data import generate_offline_data
from bb_pipeline.features import build_features, feature_names, compute_reward, reward_to_outcome_label
from bb_pipeline.blackbox import train_blackbox, collect_representations, auto_device
from bb_pipeline.sae import train_sae, encode_all
from bb_pipeline.evidence import collect_evidence, render_evidence_block
from bb_pipeline.llm import StubClient, QwenLocalClient, get_default_client
from bb_pipeline.llm1_concept import propose_concept
from bb_pipeline.verifier import score_concept, label_with_refinement
from bb_pipeline.llm2_explainer import explain_one
from bb_pipeline.judge import judge_explanation, kl_blackbox_judge
from bb_pipeline.pipeline import explain_with_selection

CONFIG = dict(
    n=10000,         # POLAR trajectories
    p=0.95,          # behavior-policy quality (0.5 random, 1.0 fully optimal)
    train_frac=0.8,
    mlp_hidden=64,
    mlp_dropout=0.1,
    mlp_weight_decay=1e-3,
    mlp_epochs=300,
    sae_latent=32,
    sae_k=8,
    sae_epochs=150,
    n_pos_train=5, n_neg_train=5,    # evidence shown to LLM1 (lean for speed)
    n_pos_val=8,   n_neg_val=8,      # held-out evidence for verifier (lean for speed)
    refinement_alpha=0.7,            # verifier-acc threshold below which we refine
    refinement_max_rounds=1,
    explain_K=4,                     # K candidates: free / balanced / low_only / high_only when polarities mixed
    n_explain_inputs=2,              # how many test inputs to explain end-to-end
    seed=0,
)
print('device =', auto_device())
print(json.dumps({k:str(v) for k,v in CONFIG.items()}, indent=2))

## 2. Generate POLAR trajectories

Each row is `(s_1, a_1, s_2, a_2, s_3, a_3, s_4)`, 11 columns. Uses POLAR's behavior policy with quality `p`.

In [ ]:
t = time.time()
D = generate_offline_data(num_samples=CONFIG['n'], p=CONFIG['p'], seed=CONFIG['seed'])
print(f'trajectories: {D.shape} in {time.time()-t:.1f}s')
print('first row:', np.round(D[0], 2))

## 3. Derived features + outcome labels

- `X` (n, 28) = derived features through stage 3, no `s_4` (because `s_4` is the future we're predicting).
- `y` (n,) = `1[reward(s_3, s_4) > median]`.

In [ ]:
X = build_features(D)
names = feature_names()
rewards = compute_reward(D)
y, threshold = reward_to_outcome_label(rewards)
print(f'X: {X.shape}  ({len(names)} named features)')
print(f'y: high/low balance = {int(y.sum())} / {len(y)}')
print(f'reward range: [{rewards.min():.2f}, {rewards.max():.2f}], threshold = {threshold:.2f}')
print('feature names:')
for i,nm in enumerate(names):
    print(f'  {i:>2d}  {nm}')

## 4. Train the MLP blackbox

Conservative sizing: `28 -> 64 -> 64 -> 2`, dropout 0.1, weight decay 1e-3, early stopping. The output `model.representation(x)` is the SAE's input.

In [ ]:
rng = np.random.default_rng(CONFIG['seed'])
idx = rng.permutation(len(X))
n_train = int(CONFIG['train_frac'] * len(X))
tr, va = idx[:n_train], idx[n_train:]

t = time.time()
model, train_res, scaler = train_blackbox(
    X[tr], y[tr], X[va], y[va],
    hidden_dim=CONFIG['mlp_hidden'], epochs=CONFIG['mlp_epochs'],
    dropout=CONFIG['mlp_dropout'], weight_decay=CONFIG['mlp_weight_decay'],
    seed=CONFIG['seed'],
)
print(f'MLP trained in {time.time()-t:.1f}s')
print(f'  train_acc={train_res.train_acc:.3f}  val_acc={train_res.val_acc:.3f}  best_epoch={train_res.best_epoch}')
print(f'  overfit_warning={train_res.overfit_warning}')
if train_res.overfit_warning:
    print('  >> consider shrinking hidden_dim to 32 and re-running.')

## 5. Train the SAE on the MLP hidden layer

Top-k SAE with auxiliary dead-feature loss (Gao et al. 2024). The number of *alive* features is bounded by the intrinsic complexity of what M actually encodes — we expect ~15 alive out of 32 for this DTR setup.

In [ ]:
H_tr = collect_representations(model, X[tr], scaler)
H_va = collect_representations(model, X[va], scaler)
print(f'hidden reps: train={H_tr.shape}, val={H_va.shape}, mean nonzero frac={(H_tr>0).mean():.3f}')

t = time.time()
sae, sae_res = train_sae(
    H_tr, H_va,
    latent_dim=CONFIG['sae_latent'], k=CONFIG['sae_k'],
    epochs=CONFIG['sae_epochs'], seed=CONFIG['seed'],
)
alive = int((sae_res.feature_density >= 0.005).sum())
print(f'SAE trained in {time.time()-t:.1f}s')
print(f'  explained_variance={sae_res.explained_variance:.3f}')
print(f'  alive features (density >= 0.5%) = {alive} / {CONFIG["sae_latent"]}')
print(f'  per-feature density distribution:')
for i, d in enumerate(sae_res.feature_density):
    bar = '█' * int(d * 30)
    tag = ' (DEAD)' if d < 0.005 else ''
    print(f'    i{i:>2d}  {d:.3f}  {bar}{tag}')

## 6. Collect evidence sets per SAE feature

For each alive feature `i` we form `(E_i^+, E_i^-)`: top activators vs. zero activators. Train half is shown to LLM1 when proposing the concept; held-out half is used by the verifier.

In [ ]:
Z_tr = encode_all(sae, H_tr)
Z_va = encode_all(sae, H_va)
X_tr_arr = X[tr]
X_va_arr = X[va]

evidence = collect_evidence(
    Z_tr,
    n_pos=CONFIG['n_pos_train'], n_neg=CONFIG['n_neg_train'],
    n_pos_val=CONFIG['n_pos_val'], n_neg_val=CONFIG['n_neg_val'],
    seed=CONFIG['seed'],
)
print(f'features with usable evidence: {len(evidence)} / {CONFIG["sae_latent"]}')
for fid in sorted(evidence)[:3]:
    e = evidence[fid]
    print(f'\n--- feature {fid} (density={e.density:.3f}) ---')
    block = render_evidence_block(
        X_tr_arr, names,
        e.pos_train_idx[:2], e.neg_train_idx[:2],
    )
    print(block)

## 7. LLM client

Default is `StubClient` — instant, deterministic, returns canned outputs that satisfy each prompt schema. Use it to verify pipeline plumbing in seconds.

To run with a real LLM, comment out the StubClient line and uncomment the QwenLocalClient line. Sizes:
- `Qwen/Qwen2.5-1.5B-Instruct` (~3 GB, safe on most GPUs)
- `Qwen/Qwen2.5-3B-Instruct` (~6 GB, better quality)
- `Qwen/Qwen2.5-7B-Instruct` (~14 GB)

In [ ]:
# LLM1/LLM2/Judge client. Auto-pick a sensible model:
#   - On Colab T4 (16GB CUDA)  → Qwen2.5-7B-Instruct  (best quality)
#   - Locally on Apple MPS / smaller GPUs → Qwen2.5-3B-Instruct
#   - Override by editing the lines below if you want StubClient or DashScope.
import torch
from bb_pipeline.verifier import EmbeddingVerifier, LLMVerifier

if IN_COLAB and torch.cuda.is_available():
    QWEN_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
    QWEN_DEVICE = 'cuda'
elif torch.cuda.is_available():
    QWEN_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
    QWEN_DEVICE = 'cuda'
else:
    QWEN_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
    QWEN_DEVICE = None  # let QwenLocalClient auto-detect (MPS / CPU)

client = QwenLocalClient(model_id=QWEN_MODEL, device=QWEN_DEVICE)
# client = StubClient()  # uncomment for an instant dry-run with no LLM
verifier = EmbeddingVerifier(model_name='sentence-transformers/all-MiniLM-L6-v2')

print(f'LLM1/LLM2/Judge client: {client.name}  ({QWEN_MODEL})')
print(f'verifier backend: {type(verifier).__name__}')
_ = client.chat([{'role':'user','content':'hi'}], max_new_tokens=4)
verifier._ensure_loaded()
print('all models warmed up')

## 7b. (Optional) Resume from a saved snapshot — skip the 17-min LLM1 phase

If you've previously run the full pipeline at least once and have a
snapshot at `snapshot/run_001/` (locally) or
`/content/drive/MyDrive/bb_pipeline_snapshots/run_001/` (Colab), you can
**skip cells 4-16** and load the expensive intermediate state directly.

Workflow when iterating on LLM2 / Judge / loss only:

1. Run cell 0 (Colab setup)
2. Run cell 2 (imports + CONFIG)
3. Run cell 14 (Qwen client)
4. Set `USE_SNAPSHOT = True` in the next cell and run it
5. Skip cells 4-16 entirely (they're not needed)
6. Jump to cell 18 (LLM2 + Judge)

Useful when changing things like the LLM2 prompt, K-candidate strategy,
loss weights, polarity thresholds, etc. NOT useful when changing the
features, MLP architecture, SAE size, or LLM1 prompts — those need a
full re-run.

In [ ]:
# === Optional: load snapshot ===
# Toggle this to True to skip cells 4-16 and resume from disk.
USE_SNAPSHOT = False

if USE_SNAPSHOT:
    from bb_pipeline.state_io import load_pipeline_state
    from bb_pipeline.blackbox import OutcomeMLP
    from bb_pipeline.sae import TopKSAE
    from bb_pipeline.feature_polarity import render_polarity_tag

    snapshot_dir = (
        '/content/drive/MyDrive/bb_pipeline_snapshots/run_001'
        if IN_COLAB else 'snapshot/run_001'
    )
    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')

    state = load_pipeline_state(snapshot_dir, mlp_class=OutcomeMLP, sae_class=TopKSAE)

    # Restore every variable downstream cells (18, 20) expect.
    model = state['model']
    sae = state['sae']
    scaler = state['scaler']
    X = state['X']
    y = state['y']
    tr = state['tr_idx']
    va = state['va_idx']
    H_tr, H_va = state['H_tr'], state['H_va']
    Z_tr, Z_va = state['Z_tr'], state['Z_va']
    X_tr_arr, X_va_arr = X[tr], X[va]
    evidence = state['evidence']
    concept_labels = state['concept_labels']
    polarities = state['polarities']
    names = feature_names()
    rng = np.random.default_rng(CONFIG['seed'])  # cell 18's test_pick uses this

    print(f'✓ Loaded snapshot from {snapshot_dir}')
    print(f'  {len(concept_labels)} labeled concepts, {len(polarities)} polarities')
    print(f'  device: {next(model.parameters()).device}')
    print('\nNow skip directly to cell 18 (LLM2 + Judge).')
else:
    print('USE_SNAPSHOT = False — proceed with the full pipeline (cells 4-16).')

## 8. LLM1: propose concept labels with a TWO-PASS gold-buffer loop

PDF Step 6 (final paragraph) describes "gold buffer" — store high-quality (concept, evidence) records and prepend them as few-shot demonstrations to subsequent LLM1 calls.

Our two-pass implementation:

1. **Pass 1**: name every alive feature with no buffer (cold start). Concepts that hit `verifier_acc >= alpha` are pushed to the gold buffer.
2. **Pass 2**: for the remaining low-acc features, re-run LLM1 with `gold_k` worked examples from pass 1 as few-shot context, hoping the style guidance produces a better concept.

Final concept dict combines pass-1 winners + pass-2 (re)labels for the rest.

In [ ]:
from bb_pipeline.gold_buffer import GoldBuffer

concept_labels = {}     # {feature_idx: concept string}
concept_records = {}    # {feature_idx: ConceptResult}
verifier_records = {}   # {feature_idx: VerifierResult}

# === PASS 1: cold start, no gold buffer ===
print('=== PASS 1: cold-start naming (no gold buffer) ===')
t0 = time.time()
gold = GoldBuffer(max_size=10)
pass1_failures = []
for fid in sorted(evidence):
    res, vres = label_with_refinement(
        client, evidence[fid], X_tr_arr, names,
        verifier=verifier,
        alpha=CONFIG['refinement_alpha'],
        max_rounds=CONFIG['refinement_max_rounds'],
    )
    concept_labels[fid] = res.concept
    concept_records[fid] = res
    verifier_records[fid] = vres
    tag = ''
    if vres.accuracy >= CONFIG['refinement_alpha']:
        gold.push(evidence[fid], res.concept, vres.accuracy)
        tag = '  -> pushed to gold buffer'
    else:
        pass1_failures.append(fid)
    print(f'  i{fid}: "{res.concept}"  acc={vres.accuracy:.2f}  rounds={len(res.refinement_history)-1}{tag}')

print(f'\nPass 1 done in {time.time()-t0:.1f}s')
print(f'Gold buffer size: {len(gold)}')
print(f'Pass 2 will re-label {len(pass1_failures)} features (acc < {CONFIG["refinement_alpha"]}).')

# === PASS 2: re-label failures with few-shot context from gold buffer ===
print('\n=== PASS 2: re-naming low-acc features with gold-buffer few-shot ===')
t1 = time.time()
pass2_improved = 0
for fid in pass1_failures:
    if len(gold) == 0:
        print(f'  i{fid}: skipped (gold buffer empty, no examples to learn from)')
        continue
    new_res, new_vres = label_with_refinement(
        client, evidence[fid], X_tr_arr, names,
        verifier=verifier,
        alpha=CONFIG['refinement_alpha'],
        max_rounds=CONFIG['refinement_max_rounds'],
        gold_buffer=gold, gold_k=2, gold_seed=fid,   # deterministic per feature
    )
    old_acc = verifier_records[fid].accuracy
    delta = new_vres.accuracy - old_acc
    keep = new_vres.accuracy >= old_acc
    keep_tag = 'kept' if keep else 'reverted'
    print(f'  i{fid}: "{new_res.concept}"  acc={new_vres.accuracy:.2f}  (was {old_acc:.2f}, Δ={delta:+.2f}, {keep_tag})')
    if keep:
        concept_labels[fid] = new_res.concept
        concept_records[fid] = new_res
        verifier_records[fid] = new_vres
        if new_vres.accuracy >= CONFIG['refinement_alpha']:
            pass2_improved += 1

print(f'\nPass 2 done in {time.time()-t1:.1f}s')
print(f'Features pushed above alpha by gold buffer: {pass2_improved} / {len(pass1_failures)}')
print(f'\nTOTAL LLM1 time: {time.time()-t0:.1f}s')
print(f'Final concept count above alpha: {sum(1 for v in verifier_records.values() if v.accuracy >= CONFIG["refinement_alpha"])} / {len(verifier_records)}')

## 9. Pick test inputs and run end-to-end (LLM2 + Judge + K-selection)

For each chosen input from the validation set:
1. Get blackbox prediction `P_M(y | x)`.
2. Get active SAE features and their values.
3. Generate K candidate explanations (LLM2).
4. Score each with the Judge (KL + sparsity + length penalties).
5. Print the winning explanation.

In [ ]:
## 8b. Compute SAE feature directionality (polarity)
# For each labeled SAE feature, what does its activation say about the
# blackbox label distribution? We use point-biserial correlation between
# (z_i > 0) and y on the train set — see feature_polarity.py.
#
# This is then handed to LLM2 so it can write directionally faithful
# explanations (e.g., not claim a "[→ LOW]" feature supports HIGH outcome).
from bb_pipeline.feature_polarity import compute_polarities, render_polarity_tag

polarities = compute_polarities(Z_tr, y[tr], feature_indices=sorted(concept_labels.keys()))
print(f'computed polarity for {len(polarities)} features')
print(f'  HIGH-pushing: {sum(1 for p in polarities.values() if p.direction == "HIGH")}')
print(f'  LOW-pushing:  {sum(1 for p in polarities.values() if p.direction == "LOW")}')
print(f'  NEUTRAL:      {sum(1 for p in polarities.values() if p.direction == "NEUTRAL")}')

print('\nSample annotated dictionary entries:')
for fid in sorted(polarities)[:5]:
    print(f'  i{fid} -> "{concept_labels[fid]}" {render_polarity_tag(polarities[fid])}')

In [ ]:
## 8c. Snapshot the expensive state to disk
# After this cell you can quick-iterate the LLM2/Judge/loss side by reloading
# from disk instead of re-running the ~17min LLM1 phase.
#
# On Colab the snapshot is written to Google Drive so it survives runtime
# disconnects. Locally it goes to ./snapshot/.
from bb_pipeline.state_io import save_pipeline_state

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    snapshot_dir = '/content/drive/MyDrive/bb_pipeline_snapshots/run_001'
else:
    snapshot_dir = 'snapshot/run_001'

save_pipeline_state(
    snapshot_dir,
    config=CONFIG,
    traj=D, X=X, y=y, tr_idx=tr, va_idx=va, scaler=scaler,
    model=model, sae=sae,
    H_tr=H_tr, H_va=H_va, Z_tr=Z_tr, Z_va=Z_va,
    evidence=evidence,
    concept_labels=concept_labels,
    verifier_records=verifier_records,
    polarities=polarities,
)
# To resume later (skipping cells 4-16):
#   from bb_pipeline.state_io import load_pipeline_state
#   from bb_pipeline.blackbox import OutcomeMLP
#   from bb_pipeline.sae import TopKSAE
#   state = load_pipeline_state(snapshot_dir,
#                               mlp_class=OutcomeMLP, sae_class=TopKSAE)

In [ ]:
import torch
device = next(model.parameters()).device
test_pick = rng.choice(len(va), size=CONFIG['n_explain_inputs'], replace=False)

results = []
for k_, vidx in enumerate(test_pick):
    x = X_va_arr[vidx]
    z = Z_va[vidx]
    active = [int(j) for j in np.where(z > 0)[0] if j in concept_labels]
    if not active:
        print(f'[skip {k_}] no labeled active features for this input')
        continue
    activ_vals = [float(z[j]) for j in active]
    x_norm = ((x - scaler['mean']) / scaler['std']).astype(np.float32)
    with torch.no_grad():
        p_m_raw = model.predict_proba(torch.from_numpy(x_norm).to(device).unsqueeze(0)).cpu().numpy().reshape(-1)
    p_m = np.array([float(p_m_raw[0]), float(p_m_raw[1])])
    print(f'\n=========================================================')
    print(f'input #{k_} (val idx {vidx})')
    print(f'  blackbox P(low, high) = ({p_m[0]:.3f}, {p_m[1]:.3f})')
    print(f'  active features = {active}')
    # Show polarities for the active features so we can sanity-check
    for j in active:
        if j in polarities:
            p = polarities[j]
            print(f'    i{j}: "{concept_labels[j]}" {render_polarity_tag(p)}')
    res = explain_with_selection(
        client, input_idx=int(vidx),
        p_blackbox=p_m,
        active_features=active, activation_values=activ_vals,
        concept_labels=concept_labels,
        polarities=polarities,
        K=CONFIG['explain_K'],
    )
    for c_i, c in enumerate(res.candidates):
        marker = '<-- WIN' if c_i == res.winner_index else ''
        print(f'  cand {c_i}: KL={c.kl:.3f}  |S|={c.sparsity}  Len={c.length}  total={c.total_loss:.3f}  Judge=({c.judge.prob_low:.2f},{c.judge.prob_high:.2f}) {marker}')
    print('  WINNING EXPLANATION:')
    print('  ' + res.winner.explanation.raw_text.replace('\n', '\n  '))
    results.append(res)

## 10. Summary

In [ ]:
if results:
    kls = [r.winner.kl for r in results]
    print(f'#explained inputs: {len(results)}')
    print(f'mean winning KL(P_M || P_Judge): {np.mean(kls):.3f}')
    print(f'mean cited features per winner: {np.mean([r.winner.sparsity for r in results]):.1f}')
    print(f'mean mechanism count per winner: {np.mean([r.winner.length for r in results]):.1f}')

    # Verifier-acc summary
    accs = [v.accuracy for v in verifier_records.values()]
    print(f'\nverifier accuracy across {len(accs)} concepts: mean={np.mean(accs):.2f}, min={np.min(accs):.2f}, max={np.max(accs):.2f}')
    print(f'concepts above alpha={CONFIG["refinement_alpha"]}: {sum(a >= CONFIG["refinement_alpha"] for a in accs)} / {len(accs)}')
else:
    print('no inputs explained -- check that some active features have concept labels')

## Next steps you might want

- **Switch to a real LLM** (cell 7) to see meaningful concept names and explanations.
- **Increase `n_explain_inputs`** for a real KL distribution.
- **Add a gold buffer** that stores winning records and prepends them as few-shot examples in subsequent LLM1 calls (PDF Step 6 final paragraph).
- **Try harder tasks**: drop more features (e.g., also drop `s_3` to predict from earlier history only) to push the MLP harder and produce more diverse SAE concepts.